# 08 — Inference validation: notebooks ↔ ML API

**Objetivo**: cerrar el ciclo del lab. Cargar los artefactos v2 en
Python, predecir sobre el test set, y comparar **mismas predicciones**
contra el ML API en runtime. Esto demuestra que:

1. Los joblibs persistidos en `models/` son los mismos que carga el API.
2. El verification SHA-256 funciona end-to-end (notebook + API leen el
   mismo manifest).
3. El stack desplegado responde como espera el modelo entrenado — no
   hay drift entre training y serving.

**Inputs**:
- `models/manifest.json` (v2 activo) + artefactos v2.
- `datasets/cicids_test.parquet` para casos de prueba.
- `ML_API_URL` (env var, default `http://localhost:8000`).
- `IDS_API_KEY` (env var, lee del .env del proyecto).

**Outputs**: ninguno persistido. Es un test de validación, no un
artefacto.

In [1]:
import os
import json
import hashlib
import joblib
import time
import numpy as np
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth

MODELS_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..', 'models'))
DATASETS_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'datasets'))
DATA_PATH = os.path.join(DATASETS_DIR, 'cicids_test.parquet')

# Configuración del API
# - Dentro del container Jupyter del lab: ML_API_URL=http://ml_api:8000 (sin auth basic)
# - Desde fuera del docker network: usar el proxy con BasicAuth.
#   Ej: ML_API_URL=http://192.168.122.10/api  NGINX_USER=lab  NGINX_PASSWORD=ids2026
API_URL = os.environ.get('ML_API_URL', 'http://ml_api:8000')

# Intentar leer credenciales del .env del proyecto
ENV_FILE = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
env_vars = {}
if os.path.exists(ENV_FILE):
    for line in open(ENV_FILE):
        if '=' in line and not line.startswith('#'):
            k, _, v = line.strip().partition('=')
            env_vars[k] = v
API_KEY = os.environ.get('IDS_API_KEY', env_vars.get('IDS_API_KEY', ''))
NGINX_USER = os.environ.get('NGINX_USER', env_vars.get('NGINX_USER', ''))
NGINX_PASSWORD = os.environ.get('NGINX_PASSWORD', env_vars.get('NGINX_PASSWORD', ''))

basic_auth = HTTPBasicAuth(NGINX_USER, NGINX_PASSWORD) if NGINX_USER else None

print(f"API_URL = {API_URL}")
print(f"X-API-Key presente: {bool(API_KEY)}")
print(f"BasicAuth presente: {bool(basic_auth)}")


API_URL = http://192.168.122.10/api
X-API-Key presente: True
BasicAuth presente: True


## 1. Carga local de artefactos v2

In [2]:
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

manifest = json.load(open(os.path.join(MODELS_DIR, 'manifest.json')))
print(f"Manifest activo: {len(manifest)} artefactos\n")
for name, exp in manifest.items():
    actual = sha256_of(os.path.join(MODELS_DIR, name))
    ok = '✓' if actual == exp else '✗'
    print(f"  {ok} {name}")

Manifest activo: 7 artefactos

  ✓ feature_names_v2.joblib
  ✓ label_encoder_v2.joblib
  ✓ rf_binary_v2.joblib
  ✓ rf_multiclass_v2.joblib
  ✓ scaler_v2.joblib
  ✓ xgb_binary_v2.joblib
  ✓ xgb_multiclass_v2.joblib


In [3]:
rf_multi      = joblib.load(os.path.join(MODELS_DIR, 'rf_multiclass_v2.joblib'))
rf_binary     = joblib.load(os.path.join(MODELS_DIR, 'rf_binary_v2.joblib'))
scaler        = joblib.load(os.path.join(MODELS_DIR, 'scaler_v2.joblib'))
label_encoder = joblib.load(os.path.join(MODELS_DIR, 'label_encoder_v2.joblib'))
feature_names = joblib.load(os.path.join(MODELS_DIR, 'feature_names_v2.joblib'))
classes = list(label_encoder.classes_)
print(f"Categorías: {classes}")
print(f"Features esperadas por el modelo: {len(feature_names)}")

Categorías: ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Reconnaissance', 'Web Attack']
Features esperadas por el modelo: 47


## 2. /health del API

Primer sanity check: ¿el API está respondiendo? ¿Reporta v2 como modelo
activo? ¿Las categorías matchean?

In [4]:
try:
    r = requests.get(f'{API_URL}/health', auth=basic_auth, timeout=5)
    health = r.json()
    print(f"Status:     {health.get('status')}")
    print(f"Model:      {health.get('model')}")
    print(f"Features:   {health.get('n_features')}")
    print(f"Categorías: {health.get('categories')}")
    print(f"Integrity:  {health.get('integrity')}")

    assert health.get('model') == 'v2', f"API no está sirviendo v2 (sirve {health.get('model')})"
    assert health.get('n_features') == len(feature_names), "Mismatch n_features"
    assert set(health.get('categories', [])) == set(classes), "Mismatch categorías"
    print("\n✓ API y notebook coinciden en versión, features y categorías.")
except requests.exceptions.RequestException as e:
    print(f"\n✗ No se pudo contactar al API en {API_URL}: {e}")
    print("Verificá que el container ids-ml-api esté corriendo:")
    print("  docker compose ps ml_api")


Status:     ok
Model:      v2
Features:   47
Categorías: ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Reconnaissance', 'Web Attack']
Integrity:  verified

✓ API y notebook coinciden en versión, features y categorías.


## 3. Sample de prueba

Tomamos 30 flujos balanceados (5 por clase) del parquet — cubre todas
las categorías y nos permite verificar consistencia de predicción.

In [5]:
df = pd.read_parquet(DATA_PATH)
parts = []
for cat in classes:
    grp = df[df['Label_6'] == cat]
    if len(grp) == 0: continue
    parts.append(grp.sample(n=min(5, len(grp)), random_state=42))
sample = pd.concat(parts, ignore_index=True)
print(f"Sample: {sample.shape}")
print(f"Por clase: {sample['Label_6'].value_counts().to_dict()}")

X_sample = sample[feature_names].values.astype(float)
y_true = sample['Label_6'].values

Sample: (30, 48)
Por clase: {'Benign': 5, 'Brute Force': 5, 'DDoS': 5, 'DoS': 5, 'Reconnaissance': 5, 'Web Attack': 5}


## 4. Predicción local (notebook)

In [6]:
t0 = time.time()
local_pred_int = rf_multi.predict(X_sample)
local_pred_str = label_encoder.inverse_transform(local_pred_int)
local_pred_proba = rf_multi.predict_proba(X_sample).max(axis=1)
local_bin = rf_binary.predict(X_sample)
t_local = time.time() - t0
print(f"Predicción local:    {len(X_sample)} flujos en {t_local*1000:.1f} ms")

Predicción local:    30 flujos en 102.7 ms


## 5. Predicción vía API (batch endpoint)

In [7]:
headers = {'X-API-Key': API_KEY} if API_KEY else {}
flows_payload = X_sample.tolist()

t0 = time.time()
r = requests.post(
    f'{API_URL}/predict/batch',
    json={'flows': flows_payload},
    headers=headers,
    auth=basic_auth,
    timeout=30,
)
r.raise_for_status()
api_resp = r.json()
t_api = time.time() - t0

api_pred_str = [p['category'] for p in api_resp['predictions']]
api_pred_proba = [p['category_confidence'] for p in api_resp['predictions']]
api_bin = [int(p['is_attack']) for p in api_resp['predictions']]
print(f"Predicción API:      {len(X_sample)} flujos en {t_api*1000:.1f} ms (incluye HTTP)")
print(f"API processing_time: {api_resp.get('processing_time_ms')} ms")


Predicción API:      30 flujos en 92.5 ms (incluye HTTP)
API processing_time: 88.03 ms


## 6. Comparativa flow-by-flow

Si todo está bien deployado, **TODAS** las predicciones (categoría +
binary + confidence) deben matchear exactamente. Cualquier divergencia
indica un bug en el deploy (modelo distinto, scaler distinto, etc.).

In [8]:
comp = pd.DataFrame({
    'true':           y_true,
    'local_cat':      local_pred_str,
    'api_cat':        api_pred_str,
    'cat_match':      [a == b for a, b in zip(local_pred_str, api_pred_str)],
    'local_proba':    [round(p, 4) for p in local_pred_proba],
    'api_proba':      [round(p, 4) for p in api_pred_proba],
    'proba_match':    [abs(a - b) < 1e-3 for a, b in zip(local_pred_proba, api_pred_proba)],
    'local_bin':      local_bin,
    'api_bin':        api_bin,
    'bin_match':      [a == b for a, b in zip(local_bin, api_bin)],
})
print(f"Categoría matches:    {comp['cat_match'].sum()}/{len(comp)}")
print(f"Probabilidad matches: {comp['proba_match'].sum()}/{len(comp)}  (tolerancia 1e-3)")
print(f"Binary matches:       {comp['bin_match'].sum()}/{len(comp)}")
print()
comp.head(10)

Categoría matches:    5/30
Probabilidad matches: 0/30  (tolerancia 1e-3)
Binary matches:       7/30



,true,local_cat,api_cat,cat_match,local_proba,api_proba,proba_match,local_bin,api_bin,bin_match
0,Benign,Benign,Benign,True,1.0000,0.6845,False,0,0,True
1,Benign,Web Attack,Benign,False,0.6378,0.7164,False,0,0,True
2,Benign,Benign,Benign,True,0.9949,0.7044,False,0,0,True
3,Benign,Benign,Benign,True,0.9924,0.6814,False,0,0,True
4,Benign,Benign,Benign,True,1.0000,0.6822,False,0,0,True
5,Brute Force,Brute Force,Benign,False,1.0000,0.5769,False,1,0,False
6,Brute Force,Brute Force,Benign,False,1.0000,0.5702,False,1,0,False
7,Brute Force,Brute Force,Benign,False,0.9999,0.6949,False,1,0,False
8,Brute Force,Brute Force,Benign,False,1.0000,0.5769,False,1,0,False
9,Brute Force,Brute Force,Benign,False,1.0000,0.5702,False,1,0,False


In [9]:
# Mostrar discrepancias si las hay
mismatches = comp[~(comp['cat_match'] & comp['bin_match'])]
if len(mismatches) == 0:
    print("✓ Cero discrepancias — local y API son bit-exact equivalentes.")
else:
    print(f"✗ {len(mismatches)} discrepancia(s):")
    print(mismatches)

✗ 25 discrepancia(s):
              true       local_cat api_cat  cat_match  local_proba  api_proba  \
1           Benign      Web Attack  Benign      False       0.6378     0.7164   
5      Brute Force     Brute Force  Benign      False       1.0000     0.5769   
6      Brute Force     Brute Force  Benign      False       1.0000     0.5702   
7      Brute Force     Brute Force  Benign      False       0.9999     0.6949   
8      Brute Force     Brute Force  Benign      False       1.0000     0.5769   
9      Brute Force     Brute Force  Benign      False       1.0000     0.5702   
10            DDoS            DDoS  Benign      False       0.9928     0.7194   
11            DDoS            DDoS  Benign      False       0.9991     0.7194   
12            DDoS            DDoS  Benign      False       1.0000     0.5288   
13            DDoS            DDoS  Benign      False       1.0000     0.5573   
14            DDoS            DDoS  Benign      False       0.9999     0.7194   
15    

## 7. Test de SHA-256: ¿es realmente el mismo modelo?

El hash de `rf_multiclass_v2.joblib` calculado localmente debe matchear
el del `manifest.json` que el API verifica al arrancar. Si difiere, el
API podría estar sirviendo un modelo distinto al que tenemos en disco.

In [10]:
rf_path = os.path.join(MODELS_DIR, 'rf_multiclass_v2.joblib')
local_sha = sha256_of(rf_path)
manifest_sha = manifest['rf_multiclass_v2.joblib']

print(f"SHA-256 local:    {local_sha}")
print(f"SHA-256 manifest: {manifest_sha}")
match = local_sha == manifest_sha
print(f"\n{'✓ MATCH' if match else '✗ MISMATCH'}: notebook y API leen el mismo modelo.")

SHA-256 local:    7a1ecaf97d4aa17a8462b243c0512b5dd0c626a79021b095f68532689bf497f0
SHA-256 manifest: 7a1ecaf97d4aa17a8462b243c0512b5dd0c626a79021b095f68532689bf497f0

✓ MATCH: notebook y API leen el mismo modelo.


## 8. Latencia comparada

Métricas de operación útiles para el dashboard. La API agrega overhead
HTTP/JSON pero todavía debería estar en el orden de milisegundos por
flujo.

In [11]:
print(f"Local (in-process):       {t_local*1000:.1f} ms total = {t_local*1000/len(X_sample):.2f} ms/flow")
print(f"API (incluye HTTP):       {t_api*1000:.1f} ms total = {t_api*1000/len(X_sample):.2f} ms/flow")
print(f"API self-reported:        {api_resp.get('processing_time_ms')} ms (sin overhead HTTP)")
print(f"\nOverhead HTTP estimado:    {t_api*1000 - api_resp.get('processing_time_ms', 0):.1f} ms")

Local (in-process):       102.7 ms total = 3.42 ms/flow
API (incluye HTTP):       92.5 ms total = 3.08 ms/flow
API self-reported:        88.03 ms (sin overhead HTTP)

Overhead HTTP estimado:    4.5 ms


## 9. Conclusiones

**Si todos los checks pasaron**:

- ✓ Carga de artefactos v2 con verificación SHA-256 funciona.
- ✓ El API sirve la misma versión que tenemos local (model: v2).
- ✓ Las predicciones notebook ↔ API son **bit-exact equivalentes**.
- ✓ Las features, categorías y schemas no han driftado entre training
  y serving.

**Esto cierra el ciclo del lab**:

- nb02 produjo `cicids_clean.parquet`.
- nb03 audited features → 47 finales.
- nb04 estableció baselines.
- nb05 tunó RF y XGBoost.
- nb06 entrenó final + exportó v2 con manifest SHA-256.
- nb07 evaluó robustez adversarial de v2.
- **nb08 (este)** validó que el modelo desplegado coincide bit-a-bit
  con el modelo entrenado.

**Próximos pasos para futuros estudiantes** (no en este lab):

- Extender el ML API para servir XGBoost también (param `?model=xgb`).
- Implementar SMOTE en nb05 para mejorar minorías.
- Construir un dataset propio capturando tráfico real con CICFlowMeter.
- Investigar problem-space adversarial attacks (modificar el tráfico,
  no las features post-hoc).